# Module 9 — Advanced Retrieval Engineering

Colab-ready lab: lexical retrieval → dense retrieval → hybrid RRF → filtering → reranking → evaluation → failure injection.

In [ ]:
import math, re
from collections import Counter
TOKEN_RE=re.compile(r'[A-Za-z0-9_./:-]+')
def tok(s): return [x.lower() for x in TOKEN_RE.findall(s)]

In [ ]:
docs={
'd1':'CICS transaction timeout APAR PH12345 requires region restart',
'd2':'Restart the CICS region after correcting timeout configuration',
'd3':'AWS IAM troubleshooting for an expired access key',
'd4':'Transaction processing is interrupted when a regional service exceeds its timeout threshold'
}
def bm25(query,k=5):
    q=tok(query); ds={i:tok(t) for i,t in docs.items()}; avg=sum(map(len,ds.values()))/len(ds); df=Counter()
    for terms in ds.values(): df.update(set(terms))
    out=[]
    for i,terms in ds.items():
        tf=Counter(terms); score=0
        for term in q:
            if term in tf:
                idf=math.log(1+(len(ds)-df[term]+.5)/(df[term]+.5))
                score += idf*(tf[term]*2.5)/(tf[term]+1.5*(.25+.75*len(terms)/avg))
        out.append((i,score))
    return sorted(out,key=lambda x:x[1],reverse=True)[:k]
print(bm25('PH12345'))

In [ ]:
def embed(text,d=64):
    v=[0.0]*d
    for x in tok(text): v[hash(x)%d]+=1
    n=math.sqrt(sum(z*z for z in v)) or 1
    return [z/n for z in v]
def cosine(a,b): return sum(x*y for x,y in zip(a,b))
def dense(query,k=5):
    q=embed(query)
    return sorted([(i,cosine(q,embed(t))) for i,t in docs.items()],key=lambda x:x[1],reverse=True)[:k]
print(dense('transaction processing timeout'))

In [ ]:
def rrf(*lists, rrf_k=60, limit=5):
    s={}
    for ranked in lists:
        for rank,(doc,_) in enumerate(ranked,1): s[doc]=s.get(doc,0)+1/(rrf_k+rank)
    return sorted(s.items(),key=lambda x:x[1],reverse=True)[:limit]
hybrid=rrf(bm25('CICS timeout',5),dense('CICS timeout',5))
print(hybrid)

## Exercises
1. Change the query to an exact error code and compare lexical vs dense retrieval.
2. Add 20 synthetic documents and create 10 labeled queries.
3. Sweep `rrf_k` across 1, 10, 30, 60, 100.
4. Add tenant metadata and implement pre-retrieval authorization filtering.
5. Add a reranking score using query-token overlap.
6. Calculate Recall@1/5 and MRR for your labeled queries.
7. Intentionally remove the only relevant candidate and explain why reranking cannot recover it.
8. Rewrite a precise query into a broad query and measure retrieval drift.
9. Compress a 10-document evidence set while preserving provenance.
10. Write a recommendation for the minimum sufficient production retrieval architecture.

In [ ]:
def recall_at_k(ranked,relevant,k): return len(set(x[0] for x in ranked[:k]) & set(relevant))/max(len(relevant),1)
def mrr(ranked,relevant):
    for i,(doc,_) in enumerate(ranked,1):
        if doc in relevant: return 1/i
    return 0
ranked=bm25('PH12345',5)
print('Recall@1:',recall_at_k(ranked,{'d1'},1),'MRR:',mrr(ranked,{'d1'}))

## Failure injection checklist
- lexical-only paraphrase miss
- dense-only identifier miss
- incomparable-score averaging
- filter-after-retrieval leakage
- insufficient reranker candidate pool
- query rewrite drift
- evidence compression deleting a qualifying clause

**Mastery:** explain every ranking change rather than merely observing it.